# MimicryDB-Auto: Statistical Analysis Pipeline

**Paper:** MimicryDB-Auto: A Curated Multi-Pathogen Dataset of Structurally Validated Pathogen-Host Peptide Pairs  

---

## Notebook Structure
1. Setup & Data Loading
2. Data Preparation
3. Core Statistical Analyses (Section 3.2)
4. Original RF Classifier — Y vs N at 2.0 Å (Section 3.3)
5. Strong vs Weak Mimic RF — within Y-class at 1.0 Å (Section 3.3 / 4.3)
6. Threshold Sensitivity Analysis (Section 3.4)
7. Figures

---
## SECTION 1: Setup & Data Loading

In [21]:
# ─────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, mannwhitneyu, norm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, recall_score, precision_score,
                             accuracy_score, classification_report,
                             confusion_matrix, roc_curve)
from matplotlib.patches import Patch

print('All libraries loaded successfully')

All libraries loaded successfully


In [22]:
# ── Load dataset ───────────────────────────────────────────────────
df = pd.read_excel('/content/ML targets (10).xlsx', sheet_name='Sheet1')
print('Shape:', df.shape)
print('\nLabel counts:')
print(df['RMSD_Mimic_Target (Y)'].value_counts())
print('\nColumns:', df.columns.tolist())

Shape: (399, 25)

Label counts:
RMSD_Mimic_Target (Y)
Y    262
N    137
Name: count, dtype: int64

Columns: ['Organism', 'Protein', 'Position', 'HLA Haplotype', 'Pathogen Peptide', 'pathogen length', '%Rank_EL(X)', 'Aff(nM)(X)', 'Immunogenicity', 'Type of MHC', 'Human_match', 'BLOSUM80 score', 'Identity percentage', 'Alignment length (Sequence)', 'Identical aa', 'Positions', 'Human Peptide', 'Human length', 'Alignment Length (Structure)', 'Structural RMSD', 'TM-align score (Human chain 2)', 'Structural alignment coverage %', 'RMSD_Mimic_Target (Y)', 'BLOSUM80_per_residue ', 'Alignment_coverage_sequence']


---
## SECTION 2: Data Preparation

This section comes before anything else. It converts types, computes derived features, and defines all class subgroups used throughout.

In [23]:
# ── Type conversion + derived features ─────────────────────────────
# Converting all numeric columns from object type
for col in ['BLOSUM80 score', 'Identity percentage', 'Alignment length (Sequence)',
            'Identical aa', 'pathogen length', 'Human length', '%Rank_EL(X)',
            'Aff(nM)(X)', 'Structural RMSD', 'TM-align score (Human chain 2)',
            'Structural alignment coverage %', 'Alignment Length (Structure)']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Derived feature 1: BLOSUM80 normalised by alignment length
df['BLOSUM80_per_residue'] = df['BLOSUM80 score'] / df['Alignment length (Sequence)']

# Derived feature 2: what fraction of pathogen peptide was covered by BLAST
df['Alignment_coverage_sequence'] = (
    df['Alignment length (Sequence)'] / df['pathogen length'] * 100
).clip(upper=100)

# Fill NaN with 0 — these are genuine no-BLAST-hit entries, not missing data
df['BLOSUM80_per_residue'] = df['BLOSUM80_per_residue'].fillna(0)
df['Alignment_coverage_sequence'] = df['Alignment_coverage_sequence'].fillna(0)

print('Conversion complete. Shape:', df.shape)
print('Missing values remaining:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Conversion complete. Shape: (399, 26)
Missing values remaining:
Series([], dtype: int64)


In [24]:
# ── Defining the class subgroups ─────────────────────────────────────
# 2.0 Å binary classification
Y_group = df[df['RMSD_Mimic_Target (Y)'] == 'Y']   # n=262, RMSD < 2.0 Å
N_group = df[df['RMSD_Mimic_Target (Y)'] == 'N']   # n=137, RMSD >= 2.0 Å

# At 1.0 Å
strong_mimics = df[df['Structural RMSD'] < 1.0]
weak_mimics   = df[(df['Structural RMSD'] >= 1.0) & (df['Structural RMSD'] < 2.0)]
non_mimics    = df[df['Structural RMSD'] >= 2.0]

# Consistent colour scheme used throughout all figures
strong_color = '#1f78b4'   # deep blue
weak_color   = '#e6ab02'   # golden
non_color    = '#d95f02'   # orange-red

print(f'Y_group (RMSD < 2.0 Å):  n={len(Y_group)}')
print(f'N_group (RMSD >= 2.0 Å): n={len(N_group)}')
print(f'Strong mimics (< 1.0 Å): n={len(strong_mimics)}')
print(f'Weak mimics (1.0-2.0 Å): n={len(weak_mimics)}')
print(f'Non-mimics  (>= 2.0 Å):  n={len(non_mimics)}')

Y_group (RMSD < 2.0 Å):  n=262
N_group (RMSD >= 2.0 Å): n=137
Strong mimics (< 1.0 Å): n=159
Weak mimics (1.0-2.0 Å): n=103
Non-mimics  (>= 2.0 Å):  n=137


---
## SECTION 4: Original RF Classifier — Y vs N at 2.0 Å

This is the primary RF reported in Section 3.3 of the paper. The N-class has zero sequence values by construction, making this an upper-bound estimate of classifier performance.

In [25]:
# ──Feature list for all RF analyses ───────────────────────────────
FEATURES = [
    'BLOSUM80 score',
    'Identity percentage',
    'Alignment length (Sequence)',
    'Identical aa',
    'pathogen length',
    'Human length',
    '%Rank_EL(X)',
    'Aff(nM)(X)',
    'Structural alignment coverage %',
    'BLOSUM80_per_residue',
    'Alignment_coverage_sequence'
    # NOTE: Structural RMSD and TM-align score excluded to prevent label leakage
]
print(f'Feature count: {len(FEATURES)}')
print(FEATURES)

Feature count: 11
['BLOSUM80 score', 'Identity percentage', 'Alignment length (Sequence)', 'Identical aa', 'pathogen length', 'Human length', '%Rank_EL(X)', 'Aff(nM)(X)', 'Structural alignment coverage %', 'BLOSUM80_per_residue', 'Alignment_coverage_sequence']


In [26]:
# ──Train/test split + RF training (Y vs N, 2.0 Å) ────────────────
X = df[FEATURES].fillna(0)   # zeros = genuine no-BLAST-hit, not imputed missing
y = (df['RMSD_Mimic_Target (Y)'] == 'Y').astype(int)

print(f'X shape: {X.shape}')
print(f'Missing values: {X.isnull().sum().sum()}  (should be 0)')
print(f'N-class entries with non-zero BLAST: {((df["RMSD_Mimic_Target (Y)"]=="N") & (df["BLOSUM80 score"]>0)).sum()}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f'\nTrain: {X_train.shape[0]} (Y={y_train.sum()}, N={(y_train==0).sum()})')
print(f'Test:  {X_test.shape[0]}  (Y={y_test.sum()}, N={(y_test==0).sum()})')

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_split=5,
    min_samples_leaf=2, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
print('\nModel trained successfully')

X shape: (399, 11)
Missing values: 0  (should be 0)
N-class entries with non-zero BLAST: 9

Train: 319 (Y=209, N=110)
Test:  80  (Y=53, N=27)

Model trained successfully


In [27]:
# ──Evaluate RF (Y vs N, 2.0 Å) ───────────────────────────────────
y_pred       = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
auc          = roc_auc_score(y_test, y_pred_proba)
cm           = confusion_matrix(y_test, y_pred)

print('=== RF Performance: Y vs N at 2.0 Å (held-out test set, n=80) ===')
print(classification_report(y_test, y_pred, target_names=['Non-Mimic (N)', 'Confirmed Mimic (Y)']))
print(f'AUC-ROC: {auc:.3f}')
print(f'Confusion matrix:\n{cm}')

=== RF Performance: Y vs N at 2.0 Å (held-out test set, n=80) ===
                     precision    recall  f1-score   support

      Non-Mimic (N)       1.00      0.85      0.92        27
Confirmed Mimic (Y)       0.93      1.00      0.96        53

           accuracy                           0.95        80
          macro avg       0.96      0.93      0.94        80
       weighted avg       0.95      0.95      0.95        80

AUC-ROC: 0.958
Confusion matrix:
[[23  4]
 [ 0 53]]


In [28]:
# ──5-fold CV for original RF (Y vs N) ────────────────────────────
cv_orig = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_orig = cross_val_score(rf_model, X, y, cv=cv_orig, scoring='roc_auc')

print('=== 5-Fold CV: Y vs N at 2.0 Å ===')
print(f'AUC-ROC: {cv_scores_orig.mean():.3f} ± {cv_scores_orig.std():.3f}')
print(f'Per-fold: {[f"{s:.3f}" for s in cv_scores_orig]}')

=== 5-Fold CV: Y vs N at 2.0 Å ===
AUC-ROC: 0.979 ± 0.018
Per-fold: ['0.999', '0.959', '0.979', '0.957', '1.000']


In [29]:
# ──Feature importances ───────────────────────────────────────────
importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('=== Feature Importances ===')
print(importance_df.to_string(index=False))

seq_features = ['BLOSUM80 score', 'Identity percentage', 'Alignment length (Sequence)',
                'Identical aa', 'BLOSUM80_per_residue', 'Alignment_coverage_sequence']
seq_importance = importance_df[importance_df['Feature'].isin(seq_features)]['Importance'].sum()
print(f'\nSequence features total importance: {seq_importance:.3f} ({seq_importance*100:.1f}%)')

=== Feature Importances ===
                        Feature  Importance
    Alignment length (Sequence)    0.202925
    Alignment_coverage_sequence    0.189358
           BLOSUM80_per_residue    0.180776
                   Identical aa    0.154240
                 BLOSUM80 score    0.114122
            Identity percentage    0.112680
                   Human length    0.015819
Structural alignment coverage %    0.012940
                    %Rank_EL(X)    0.010126
                     Aff(nM)(X)    0.005082
                pathogen length    0.001933

Sequence features total importance: 0.954 (95.4%)


---
## SECTION 5: Strong vs Weak Mimic RF

Both classes here passed identical sequence thresholds (>=50% identity, >=90% coverage).

In [30]:
# ──Build strong vs weak mimic dataset ────────────────────────────
# Take only original Y-class pairs (all sequence-similar)
y_class_pairs = df[df['RMSD_Mimic_Target (Y)'] == 'Y'].copy()

# Label: 1=strong mimic (RMSD<1.0), 0=weak mimic (1.0<=RMSD<2.0)
y_class_pairs['strong_mimic'] = (y_class_pairs['Structural RMSD'] < 1.0).astype(int)

# To drop rows with missing feature data
ml_df = y_class_pairs.dropna(subset=FEATURES)

print('=== Strong vs Weak Mimic Dataset ===')
print(f'Total Y-class pairs with complete data: {len(ml_df)}')
print(f'Strong Mimics (RMSD < 1.0 Å):           {ml_df["strong_mimic"].sum()}')
print(f'Weak Mimics   (1.0 <= RMSD < 2.0 Å):    {(ml_df["strong_mimic"]==0).sum()}')
print()
print('Both classes passed >=50% identity threshold — no categorical sequence separation.')
print('This is the biologically realistic classification task.')

=== Strong vs Weak Mimic Dataset ===
Total Y-class pairs with complete data: 262
Strong Mimics (RMSD < 1.0 Å):           159
Weak Mimics   (1.0 <= RMSD < 2.0 Å):    103

Both classes passed >=50% identity threshold — no categorical sequence separation.
This is the biologically realistic classification task.


In [31]:
# ──Training RF (strong vs weak) ─────────────────────────────────────
X_sw = ml_df[FEATURES]
y_sw = ml_df['strong_mimic']

X_sw_train, X_sw_test, y_sw_train, y_sw_test = train_test_split(
    X_sw, y_sw, test_size=0.2, stratify=y_sw, random_state=42)

print(f'Train: {X_sw_train.shape[0]}')
print(f'Test:  {X_sw_test.shape[0]}  (Strong={y_sw_test.sum()}, Weak={(y_sw_test==0).sum()})')

rf_sw = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_split=5,
    min_samples_leaf=2, class_weight='balanced', random_state=42)
rf_sw.fit(X_sw_train, y_sw_train)

y_sw_pred  = rf_sw.predict(X_sw_test)
y_sw_proba = rf_sw.predict_proba(X_sw_test)[:, 1]
auc_sw     = roc_auc_score(y_sw_test, y_sw_proba)

from sklearn.utils import resample
aucs = []
for _ in range(1000):
    idx = resample(range(len(y_sw_test)), random_state=None)
    aucs.append(roc_auc_score(y_sw_test.iloc[idx], y_sw_proba[idx]))
print(f"Bootstrap 95% CI: {np.percentile(aucs, 2.5):.3f}–{np.percentile(aucs, 97.5):.3f}")

print(f'\n=== RF Performance: Strong vs Weak Mimics ===')
print(classification_report(y_sw_test, y_sw_pred, target_names=['Weak Mimic', 'Strong Mimic']))
print(f'AUC-ROC: {auc_sw:.3f}')
print(f'\n--- Compare to original Y vs N AUC: {auc:.3f} ---')
print(f'Drop in AUC: {auc - auc_sw:.3f}  (this quantifies inflation from categorical construction)')

Train: 209
Test:  53  (Strong=32, Weak=21)
Bootstrap 95% CI: 0.724–0.937

=== RF Performance: Strong vs Weak Mimics ===
              precision    recall  f1-score   support

  Weak Mimic       0.70      0.76      0.73        21
Strong Mimic       0.83      0.78      0.81        32

    accuracy                           0.77        53
   macro avg       0.76      0.77      0.77        53
weighted avg       0.78      0.77      0.78        53

AUC-ROC: 0.841

--- Compare to original Y vs N AUC: 0.958 ---
Drop in AUC: 0.117  (this quantifies inflation from categorical construction)


In [32]:
# ──5-fold CV for strong vs weak RF ───────────────────────────────
cv_sw = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_sw = cross_val_score(rf_sw, X_sw, y_sw, cv=cv_sw, scoring='roc_auc')

print('=== 5-Fold CV: Strong vs Weak Mimics ===')
print(f'AUC-ROC: {cv_scores_sw.mean():.3f} ± {cv_scores_sw.std():.3f}')
print(f'Per-fold: {[f"{s:.3f}" for s in cv_scores_sw]}')
print()
print(f'Held-out test AUC:  {auc_sw:.3f}')
print(f'5-fold CV AUC:      {cv_scores_sw.mean():.3f} ± {cv_scores_sw.std():.3f}')
print('Consistent results across held-out and CV confirm stability.')

=== 5-Fold CV: Strong vs Weak Mimics ===
AUC-ROC: 0.823 ± 0.053
Per-fold: ['0.746', '0.778', '0.842', '0.861', '0.888']

Held-out test AUC:  0.841
5-fold CV AUC:      0.823 ± 0.053
Consistent results across held-out and CV confirm stability.


---
## SECTION 6: Threshold Sensitivity Analysis (Table 6)

In [33]:
# ──Threshold sensitivity table ───────────────────────────────────
main_rates = {}
for thresh in [1.0, 1.5, 2.0]:
    main_rates[thresh] = (df['Structural RMSD'] < thresh).sum() / len(df)
    print(f'Main dataset mimic rate at {thresh} Å: {main_rates[thresh]:.3f} ({(df["Structural RMSD"] < thresh).sum()}/{len(df)})')

print()
print('Cross-pair background rates (from separate 136-pair validation in Section 2.5.5):')
print('  These CANNOT be computed from the main df — they come from a separate experiment.')
print('  1.0 Å: 55.9% (76/136)')
print('  1.5 Å: 85.3% (116/136)')
print('  2.0 Å: 91.9% (125/136)')
print()

# Table 6 (values from paper)
table6 = pd.DataFrame({
    'RMSD Threshold (Å)':  [1.0, 1.5, 2.0],
    'Main Dataset Rate':   [f'{main_rates[1.0]:.3f} ({(df["Structural RMSD"]<1.0).sum()}/399)',
                            f'{main_rates[1.5]:.3f} ({(df["Structural RMSD"]<1.5).sum()}/399)',
                            f'{main_rates[2.0]:.3f} ({(df["Structural RMSD"]<2.0).sum()}/399)'],
    'Cross-Pair Rate':     ['55.9% (76/136)', '85.3% (116/136)', '91.9% (125/136)'],
    'Discrimination Ratio':['0.71', '0.66', '0.71']
})
print('=== Table 6: Threshold Sensitivity ===')
print(table6.to_string(index=False))

Main dataset mimic rate at 1.0 Å: 0.398 (159/399)
Main dataset mimic rate at 1.5 Å: 0.559 (223/399)
Main dataset mimic rate at 2.0 Å: 0.657 (262/399)

Cross-pair background rates (from separate 136-pair validation in Section 2.5.5):
  These CANNOT be computed from the main df — they come from a separate experiment.
  1.0 Å: 55.9% (76/136)
  1.5 Å: 85.3% (116/136)
  2.0 Å: 91.9% (125/136)

=== Table 6: Threshold Sensitivity ===
 RMSD Threshold (Å) Main Dataset Rate Cross-Pair Rate Discrimination Ratio
                1.0   0.398 (159/399)  55.9% (76/136)                 0.71
                1.5   0.559 (223/399) 85.3% (116/136)                 0.66
                2.0   0.657 (262/399) 91.9% (125/136)                 0.71
